# Bike Rental Demand Prediction with Data Augmentation: Vanilla vs. Mixup vs. RegMix
**Dataset:** UCI Bike Sharing (`hour.csv`) only. **Primary method reference:** Hwang & Whang, *RegMix: Data Mixing Augmentation for Regression*, arXiv:2106.03374.

**How to run (Google Colab):** Runtime → Change runtime type → GPU (T4 is fine) → Runtime → Run all.
1. Run first with `PRESET = "QUICK"` (a smoke test). It checks that everything executes end to end. **Do not report QUICK numbers in the paper.**
2. Then set `PRESET = "STANDARD"` (or `"PAPER"`) and run again. Results are cached in `OUTPUT_DIR/results.jsonl`, so if the session disconnects, re-running resumes where it stopped. Set `MOUNT_DRIVE = True` to keep the cache on Google Drive.
3. Everything the paper needs (tables as CSV, figures as PNG, `config.json`) is written to `OUTPUT_DIR`.

**Two evaluation protocols (they answer different questions and are not interchangeable):**
- `paper`: a reproduction-style protocol following the RegMix paper's Table 1 for Bike: random subset, 300 training / 200 validation examples; every remaining row is the test set.
- `chrono`: a chronological split (first 60% train / next 20% validation / last 20% test) that tests generalisation to later observations.

**Documented differences from the original RegMix algorithm** (also saved in `config.json`; cite them in the paper's methodology):
1. Mixed examples are generated on the fly: at every step, an anchor example draws one partner uniformly from *its policy-selected kNN set* and a fresh λ ~ Beta(α, α). The paper's pseudocode writes `Mix(D, P)` without specifying whether all neighbour pairs are materialised. Mixup is the special case where the kNN set is all other examples.
2. Policy search uses a fixed gradient-step budget per candidate model with no early stopping; the final model uses validation-based early stopping.
3. PPO details left open in the paper are my choices: per-example probability ratio, 4 PPO epochs per iteration, clip 0.2, entropy weight 0.01 (from the paper), reward `1/val_MSE` minus an EMA baseline (0.95, from the paper), with the EMA initialised from the first batch and the advantages divided by their batch standard deviation.
4. Final policy = argmax of the converged controller.
5. The regressor learning rate is chosen on validation from a small grid (Vanilla only, to avoid favouring augmentation); the paper fixes 1e-4. The architecture, batch size, α=100, kNN options {0,1,4,16,64,256}, T=20 and the 4×100 MLP controller follow the paper.
6. The search budget is far smaller than the paper's (which reports hours per dataset), so "reduced-budget re-implementation of RegMix" is the accurate description.

In [ ]:
# 1. Imports, configuration, reproducibility
import os, sys, time, json, math, random, hashlib, zipfile, platform, warnings, urllib.request
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
try: display
except NameError: display = print
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PRESET = "QUICK"          # "QUICK" | "STANDARD" | "PAPER"
DATA_FILE = "hour.csv"    # "hour.csv" (default) or "day.csv" (sensitivity check only)
MOUNT_DRIVE = False       # True: keep outputs on Google Drive so sessions can resume
PROTOCOLS = ["paper", "chrono"]
FRACS = [0.10, 0.25, 0.50, 1.00]

PRESETS = {
    "QUICK":    dict(n_seeds=3,  regmix_iters=8,   search_steps=300,  max_steps=2000,  eval_every=100, patience=10),
    "STANDARD": dict(n_seeds=10, regmix_iters=80,  search_steps=1500, max_steps=20000, eval_every=100, patience=30),
    "PAPER":    dict(n_seeds=5,  regmix_iters=300, search_steps=3000, max_steps=40000, eval_every=200, patience=40),
}
P = PRESETS[PRESET]
CFG = dict(
    preset=PRESET, data_file=DATA_FILE, protocols=PROTOCOLS, fracs=FRACS, **P,
    hidden=[512, 256], batch=64, lr_grid=[1e-4, 3e-4, 1e-3],
    mixup_alphas=[0.2, 1.0, 10.0, 100.0],
    regmix_alpha=100.0, knn_options=[0, 1, 4, 16, 64, 256], n_policies_T=20,
    ctrl_hidden=[100, 100, 100, 100], ctrl_lr=2e-3, ppo_epochs=4, ppo_clip=0.2, entropy_w=0.01, ema=0.95,
    paper_n_train=300, paper_n_val=200, chrono_split=[0.6, 0.2, 0.2],
    chrono_regmix_max_n=2700,   # RegMix is skipped when the training subset is larger (compute); Vanilla/Mixup/GlobalkNN still run
)
SEEDS = list(range(CFG["n_seeds"]))

if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/bike_regmix_outputs"
else:
    OUT = "/content/bike_regmix_outputs" if IN_COLAB else os.environ.get("BIKE_OUT", "./bike_regmix_outputs")
for sub in ["", "figures", "tables", "preds", "data"]:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(0)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight"})
def savefig(name): plt.savefig(os.path.join(OUT, "figures", name + ".png")); plt.show()
print("device:", device, "| preset:", PRESET, "| seeds:", SEEDS, "| output:", OUT)

In [ ]:
# 2. Dataset acquisition (official UCI source) and integrity record
UCI_ZIP = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
DATA_DIR = os.environ.get("BIKE_LOCAL_DIR", os.path.join(OUT, "data"))
DATA_PATH = os.path.join(DATA_DIR, DATA_FILE)
if not os.path.exists(DATA_PATH):
    zpath = os.path.join(DATA_DIR, "bike_sharing.zip")
    try:
        urllib.request.urlretrieve(UCI_ZIP, zpath)
        zipfile.ZipFile(zpath).extractall(DATA_DIR)
    except Exception as e:
        print("Automatic download failed:", e)
        if IN_COLAB:
            from google.colab import files
            print("Download the zip from https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset, then upload hour.csv and day.csv here:")
            for fn in files.upload(): os.replace(fn, os.path.join(DATA_DIR, fn))
        else:
            raise
sha = hashlib.sha256(open(DATA_PATH, "rb").read()).hexdigest()
print(f"{DATA_FILE}: sha256={sha}")

In [ ]:
# 3. Exploratory data analysis: structure, quality checks, leakage check
raw = pd.read_csv(DATA_PATH, parse_dates=["dteday"])
df = raw.sort_values("instant").reset_index(drop=True)
HAS_HR = "hr" in df.columns
print("shape:", df.shape); print(df.dtypes.to_string())
print("\nmissing values:", int(raw.isna().sum().sum()),
      "| duplicate rows:", int(raw.duplicated().sum()),
      "| duplicates ignoring 'instant':", int(raw.drop(columns="instant").duplicated().sum()))
print("collection period:", df.dteday.min().date(), "to", df.dteday.max().date())
print("LEAKAGE CHECK  cnt == casual + registered for all rows:", bool((df.casual + df.registered == df.cnt).all()))
if HAS_HR:
    ts = df.dteday + pd.to_timedelta(df.hr, unit="h")
    gaps = (ts.diff().dropna() / pd.Timedelta(hours=1) - 1).clip(lower=0)
    print("hours absent from the time series (gaps):", int(gaps.sum()), "in", int((gaps > 0).sum()), "gaps")
print("\nTarget 'cnt': mean=%.1f std=%.1f min=%d median=%d max=%d" % (df.cnt.mean(), df.cnt.std(), df.cnt.min(), df.cnt.median(), df.cnt.max()))
print("(Compare this std with the RMSE values reported for Bike in the RegMix paper, ~369-393, when interpreting the reproduction-style protocol.)")
print("\nweathersit counts:\n", df.weathersit.value_counts().sort_index().to_string())
print("rows with hum == 0:", int((df.hum == 0).sum()), "| rows with windspeed == 0:", int((df.windspeed == 0).sum()))
print("\nDescriptive statistics:"); display(df.drop(columns=["instant", "dteday"]).describe().T.round(3))

def iqr_outliers(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return int(((s < q1 - 1.5 * i) | (s > q3 + 1.5 * i)).sum())
out_tab = pd.DataFrame({c: {"IQR_outliers": iqr_outliers(df[c]), "pct": 100 * iqr_outliers(df[c]) / len(df)} for c in ["cnt", "temp", "atemp", "hum", "windspeed"]}).T
print("\nOutlier screen (1.5xIQR rule). Decision: all rows are KEPT, because high-demand hours are genuine peaks, not errors:"); display(out_tab.round(2))
out_tab.to_csv(os.path.join(OUT, "tables", "outlier_screen.csv"))

SEASONS = {1: "Winter", 2: "Spring", 3: "Summer", 4: "Fall"}   # per official UCI documentation
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
sns.histplot(df.cnt, bins=60, ax=ax[0], color="#4C72B0"); ax[0].set(xlabel="Hourly rentals (cnt) [bikes/h]" if HAS_HR else "Daily rentals (cnt) [bikes/day]", ylabel="Count", title="(a) Raw")
sns.histplot(np.log1p(df.cnt), bins=60, ax=ax[1], color="#55A868"); ax[1].set(xlabel="log(1 + cnt)", ylabel="Count", title="(b) Log-transformed (exploration only)")
plt.tight_layout(); savefig("fig1_target_distribution")

fig, ax = plt.subplots(figsize=(7, 4))
if HAS_HR:
    d = df.assign(Season=df.season.map(SEASONS))
    sns.lineplot(data=d, x="hr", y="cnt", hue="Season", hue_order=list(SEASONS.values()), errorbar=None, ax=ax)
    ax.set(xlabel="Hour of day", ylabel="Mean rentals [bikes/h]"); ax.set_xticks(range(0, 24, 2))
else:
    sns.lineplot(data=df.assign(Season=df.season.map(SEASONS)), x="mnth", y="cnt", hue="Season", errorbar=None, ax=ax)
    ax.set(xlabel="Month", ylabel="Mean rentals [bikes/day]")
plt.tight_layout(); savefig("fig2_demand_by_hour_season")

num_cols = [c for c in ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit", "temp", "atemp", "hum", "windspeed", "cnt"] if c in df.columns]
plt.figure(figsize=(8, 6.5)); sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, cbar_kws={"label": "Pearson r"})
plt.tight_layout(); savefig("fig3_correlation_heatmap")

fig, ax = plt.subplots(1, 4, figsize=(15, 3.4))
for a, (c, lab) in zip(ax[:3], [("temp", "Normalised temperature"), ("hum", "Normalised humidity"), ("windspeed", "Normalised windspeed")]):
    a.hexbin(df[c], df.cnt, gridsize=35, cmap="viridis", mincnt=1); a.set(xlabel=lab, ylabel="Rentals")
sns.boxplot(data=df, x="weathersit", y="cnt", ax=ax[3]); ax[3].set(xlabel="Weather situation (1=clear ... 4=heavy rain/snow)", ylabel="Rentals")
plt.tight_layout(); savefig("fig_extra_weather_vs_demand")
print("Interpretation guardrail: these are marginal associations in observational data; they are not causal effects (e.g. temperature and hour-of-day are confounded).")

In [ ]:
# 4. Preprocessing (leakage-safe) and feature construction
# Removed: instant (row id), dteday (replaced by cyclical/ordinal calendar features), casual & registered (target components -> leakage).
# hr/mnth/weekday are cyclical -> sin/cos; season/weathersit one-hot (fixed categories); yr/holiday/workingday binary; weather values already normalised by the data providers.
def build_features(d):
    X = pd.DataFrame(index=d.index)
    for col, period, off in [("hr", 24, 0), ("mnth", 12, 1), ("weekday", 7, 0)]:
        if col in d.columns:
            v = d[col] - off
            X[col + "_sin"] = np.sin(2 * np.pi * v / period); X[col + "_cos"] = np.cos(2 * np.pi * v / period)
    for col in ["yr", "holiday", "workingday"]: X[col] = d[col]
    for col, cats in [("season", [1, 2, 3, 4]), ("weathersit", [1, 2, 3, 4])]:
        for c in cats: X[f"{col}_{c}"] = (d[col] == c).astype(float)
    for col in ["temp", "atemp", "hum", "windspeed"]: X[col] = d[col]
    return X.astype(float)
FEATS = build_features(df); Y = df["cnt"].values.astype(float)
print("model inputs:", FEATS.shape[1], list(FEATS.columns))

table1 = pd.DataFrame({"Item": ["File", "Source", "Rows", "Raw columns", "Collection period", "Target", "Target mean / std / max", "Model inputs (after encoding)", "Dropped columns", "Missing values", "Duplicate rows"],
    "Value": [DATA_FILE, "UCI ML Repository, Bike Sharing Dataset (id 275)", len(df), df.shape[1], f"{df.dteday.min().date()} to {df.dteday.max().date()}", "cnt (total rentals)",
              f"{df.cnt.mean():.1f} / {df.cnt.std():.1f} / {df.cnt.max()}", FEATS.shape[1], "instant, dteday, casual, registered", int(raw.isna().sum().sum()), int(raw.duplicated().sum())]})
table1.to_csv(os.path.join(OUT, "tables", "table1_dataset_summary.csv"), index=False); display(table1)

In [ ]:
# 5. Leakage-safe splitting
# paper : random 300 train / 200 val, the rest is test (reproduction-style, per RegMix Table 1 for Bike). Fresh random draw per seed.
# chrono: first 60% train pool / next 20% val / last 20% test (future-period evaluation).
# Training subsets of size frac*|pool| are drawn from the TRAINING POOL ONLY (nested across fractions within a seed).
# Scaler is fitted on the training pool only. Target standardisation uses the labelled training subset only. Metrics are computed in original bike-count units.
def get_split(protocol, frac, seed):
    rng = np.random.RandomState(1000 + seed)
    if protocol == "paper":
        nt, nv = CFG["paper_n_train"], CFG["paper_n_val"]
        perm = rng.permutation(len(df)); pool, va, te = perm[:nt], perm[nt:nt + nv], perm[nt + nv:]
        sub = pool[:int(round(frac * nt))]
    else:
        n = len(df); a, b = int(CFG["chrono_split"][0] * n), int((CFG["chrono_split"][0] + CFG["chrono_split"][1]) * n)
        pool, va, te = np.arange(a), np.arange(a, b), np.arange(b, n)
        sub = rng.permutation(pool)[:int(round(frac * a))]
    assert len(set(sub) & set(va)) == 0 and len(set(sub) & set(te)) == 0 and len(set(va) & set(te)) == 0
    if protocol == "chrono": assert pool.max() < va.min() <= va.max() < te.min()
    sc = StandardScaler().fit(FEATS.values[pool])
    f = lambda idx: torch.tensor(sc.transform(FEATS.values[idx]), dtype=torch.float32, device=device)
    mu, sd = Y[sub].mean(), Y[sub].std() + 1e-8
    g = lambda idx: torch.tensor((Y[idx] - mu) / sd, dtype=torch.float32, device=device)
    return dict(Xtr=f(sub), ytr=g(sub), Xva=f(va), yva=g(va), Xte=f(te), yte_raw=Y[te], mu=mu, sd=sd, n=len(sub), n_val=len(va), n_test=len(te))

for pr in PROTOCOLS:
    s = get_split(pr, 1.0, 0); print(f"{pr:7s} train={s['n']:6d} val={s['n_val']:6d} test={s['n_test']:6d}")

In [ ]:
# 6. Model: a batch of T independent MLPs trained in parallel (mathematically identical to T separate MLPs; used for speed)
# Architecture (paper, Table 2, Bike): [512, 256] hidden, ReLU, MSE loss, Adam, batch 64. Initialisation = PyTorch nn.Linear default.
class BatchedMLP:
    def __init__(self, T, d_in, hidden, seed):
        g = torch.Generator().manual_seed(seed); sizes = [d_in] + hidden + [1]; self.T = T; self.W, self.b = [], []
        for i, o in zip(sizes[:-1], sizes[1:]):
            bound = 1 / math.sqrt(i)
            self.W.append(((torch.rand(T, i, o, generator=g) * 2 - 1) * bound).to(device).requires_grad_())
            self.b.append(((torch.rand(T, 1, o, generator=g) * 2 - 1) * bound).to(device).requires_grad_())
    def params(self): return self.W + self.b
    def __call__(self, x):                       # x: [T, B, d] -> [T, B]
        h = x
        for k in range(len(self.W)):
            h = torch.baddbmm(self.b[k], h, self.W[k])
            if k < len(self.W) - 1: h = F.relu(h)
        return h.squeeze(-1)

@torch.no_grad()
def predict(model, X, chunk=4096):               # -> [T, N] (standardised units)
    return torch.cat([model(X[i:i + chunk].unsqueeze(0).expand(model.T, -1, -1)) for i in range(0, len(X), chunk)], 1)

def knn_table(X, kmax, chunk=2048):              # indices of the kmax nearest training neighbours (Euclidean, standardised features), self excluded
    S = X.shape[0]; out = torch.empty(S, kmax, dtype=torch.long, device=X.device)
    for i in range(0, S, chunk):
        D = torch.cdist(X[i:i + chunk], X); r = torch.arange(i, min(i + chunk, S), device=X.device)
        D[torch.arange(len(r), device=X.device), r] = float("inf")
        out[i:i + chunk] = D.topk(kmax, largest=False).indices
    return out

def train_batched(Xtr, ytr, Xva, yva, T=1, ks=None, unres=None, alpha=1.0, nbrs=None, seed=0, lr=1e-4,
                  max_steps=2000, eval_every=100, patience=10, use_best=True, batch=None):
    """Train T models simultaneously.
    ks   : LongTensor [T, S] per-example kNN count (0 = no mixing) or None for no augmentation.
    unres: BoolTensor [T]  True -> partner drawn uniformly from ALL other training examples (plain Mixup).
    Each step = `batch` original examples (+ `batch` mixed examples if augmenting; mixed samples with k=0 get weight 0).
    use_best=True: early stopping on validation MSE (patience counted in evaluations), restoring best weights per model.
    use_best=False: fixed budget, returns the final validation MSE (used inside RegMix policy search)."""
    batch = batch or CFG["batch"]; S, d = Xtr.shape
    torch.manual_seed(seed); model = BatchedMLP(T, d, CFG["hidden"], seed); opt = torch.optim.Adam(model.params(), lr=lr)
    gen = torch.Generator(device=device); gen.manual_seed(seed); aug = ks is not None
    if aug:
        a = torch.full((T,), float(alpha)) if np.isscalar(alpha) else torch.tensor(alpha, dtype=torch.float32)
        conc = a[:, None].expand(T, batch); beta = torch.distributions.Beta(conc, conc)
        unres = torch.zeros(T, dtype=torch.bool, device=device) if unres is None else unres
    best = torch.full((T,), float("inf"), device=device); bad = torch.zeros(T, device=device); best_p = [p.detach().clone() for p in model.params()]; last = None
    for step in range(1, max_steps + 1):
        idx = torch.randint(0, S, (T, batch), device=device, generator=gen); xo, yo = Xtr[idx], ytr[idx]
        if aug:
            kk = ks.gather(1, idx); u = torch.rand(T, batch, device=device, generator=gen)
            r = (u * kk).long().clamp_(max=nbrs.shape[1] - 1)
            part = torch.where(unres[:, None], (idx + torch.randint(1, S, (T, batch), device=device, generator=gen)) % S, nbrs[idx, r])
            m = (kk > 0).float(); lam = beta.sample().to(device)
            xm = lam[..., None] * xo + (1 - lam[..., None]) * Xtr[part]; ym = lam * yo + (1 - lam) * ytr[part]
            x, y, w = torch.cat([xo, xm], 1), torch.cat([yo, ym], 1), torch.cat([torch.ones_like(m), m], 1)
            loss_t = (w * (model(x) - y) ** 2).sum(1) / w.sum(1)
        else:
            loss_t = ((model(xo) - yo) ** 2).mean(1)
        opt.zero_grad(set_to_none=True); loss_t.sum().backward(); opt.step()
        if step % eval_every == 0 or step == max_steps:
            vm = ((predict(model, Xva) - yva[None]) ** 2).mean(1); last = vm; imp = vm < best
            if use_best:
                for bp, p in zip(best_p, model.params()): bp[imp] = p.detach()[imp]
            best = torch.where(imp, vm, best); bad = torch.where(imp, torch.zeros_like(bad), bad + 1)
            if use_best and bool((bad >= patience).all()): break
    if use_best:
        with torch.no_grad():
            for p, bp in zip(model.params(), best_p): p.copy_(bp)
    return model, (best if use_best else last), step

In [ ]:
# 7. RegMix: learned per-example kNN mixing policy (Algorithm 1 of the paper; deviations listed at the top of the notebook)
def regmix_search(Xtr, ytr, Xva, yva, nbrs, opts, seed, lr, iters, search_steps, T=None):
    T = T or CFG["n_policies_T"]; S = Xtr.shape[0]; K = len(opts); opts_t = torch.tensor(opts, device=device)
    torch.manual_seed(seed)
    layers, d_in = [], S
    for h in CFG["ctrl_hidden"]: layers += [nn.Linear(d_in, h), nn.ReLU()]; d_in = h
    ctrl = nn.Sequential(*layers, nn.Linear(d_in, S * K)).to(device)
    inp = torch.ones(1, S, device=device); opt = torch.optim.Adam(ctrl.parameters(), lr=CFG["ctrl_lr"]); delta = None; hist = []
    for it in range(iters):
        with torch.no_grad():
            dist = torch.distributions.Categorical(logits=ctrl(inp).view(S, K)); a = dist.sample((T,)); old_lp = dist.log_prob(a)   # a: [T, S]
        _, vmse, _ = train_batched(Xtr, ytr, Xva, yva, T=T, ks=opts_t[a], alpha=CFG["regmix_alpha"], nbrs=nbrs, seed=seed * 1000 + it, lr=lr,
                                   max_steps=search_steps, eval_every=search_steps, use_best=False)
        rew = 1.0 / vmse
        if delta is None: delta = rew.mean().detach()
        adv = []
        for t in range(T):
            adv.append(rew[t] - delta); delta = CFG["ema"] * delta + (1 - CFG["ema"]) * rew[t]
        adv = torch.stack(adv).detach(); adv = adv / (adv.std() + 1e-8)
        for _ in range(CFG["ppo_epochs"]):
            dist = torch.distributions.Categorical(logits=ctrl(inp).view(S, K)); ratio = torch.exp(dist.log_prob(a) - old_lp)
            obj = torch.min(ratio * adv[:, None], torch.clamp(ratio, 1 - CFG["ppo_clip"], 1 + CFG["ppo_clip"]) * adv[:, None]).mean()
            loss = -obj - CFG["entropy_w"] * dist.entropy().mean(); opt.zero_grad(); loss.backward(); opt.step()
        hist.append(dict(iter=it, mean_val_mse=float(vmse.mean()), best_val_mse=float(vmse.min())))
    with torch.no_grad(): final = ctrl(inp).view(S, K).argmax(-1)
    return opts_t[final], hist

In [ ]:
# 8. Experiment runner: Vanilla, Mixup (alpha tuned on validation), Global kNN (single k tuned on validation), RegMix (learned per-example policy)
RESULTS = os.path.join(OUT, "results.jsonl")
def score(yhat_std, sp):
    yhat = yhat_std * sp["sd"] + sp["mu"]; y = sp["yte_raw"]
    return dict(rmse=float(np.sqrt(mean_squared_error(y, yhat))), mae=float(mean_absolute_error(y, yhat)), r2=float(r2_score(y, yhat))), yhat

def select_lr(protocol):
    path = os.path.join(OUT, f"lr_{protocol}_{DATA_FILE}.json")
    if os.path.exists(path): return json.load(open(path))["lr"]
    sp = get_split(protocol, 1.0, SEEDS[0]); vals = []
    for lr in CFG["lr_grid"]:
        _, v, _ = train_batched(sp["Xtr"], sp["ytr"], sp["Xva"], sp["yva"], T=1, seed=SEEDS[0], lr=lr, max_steps=CFG["max_steps"], eval_every=CFG["eval_every"], patience=CFG["patience"])
        vals.append(float(v[0]))
    lr = CFG["lr_grid"][int(np.argmin(vals))]; json.dump(dict(lr=lr, grid=CFG["lr_grid"], val_mse=vals), open(path, "w")); print(protocol, "selected lr", lr, vals); return lr

def run_one(protocol, frac, seed, lr):
    sp = get_split(protocol, frac, seed); Xtr, ytr, Xva, yva = sp["Xtr"], sp["ytr"], sp["Xva"], sp["yva"]; S = sp["n"]
    kw = dict(seed=seed, lr=lr, max_steps=CFG["max_steps"], eval_every=CFG["eval_every"], patience=CFG["patience"])
    base = dict(protocol=protocol, frac=frac, n_train=S, seed=seed); rows = []; save_pred = (seed == SEEDS[0] and frac == 1.0)
    def finish(name, model, vmse, t, steps, t_idx=0, extra=None, search_time=0.0):
        sc, yhat = score(predict(model, sp["Xte"])[t_idx].cpu().numpy(), sp)
        rows.append(dict(base, method=name, **sc, val_rmse=float(np.sqrt(float(vmse[t_idx])) * sp["sd"]), time_s=t, search_time_s=search_time, steps=steps, **(extra or {})))
        if save_pred: np.save(os.path.join(OUT, "preds", f"{protocol}_{name}.npy"), yhat)
    # Vanilla
    t0 = time.time(); m, v, st = train_batched(Xtr, ytr, Xva, yva, T=1, **kw); finish("Vanilla", m, v, time.time() - t0, st)
    # Mixup: unrestricted partners; alpha grid trained in one batch, best by validation
    al = CFG["mixup_alphas"]; Ta = len(al); t0 = time.time()
    m, v, st = train_batched(Xtr, ytr, Xva, yva, T=Ta, ks=torch.ones(Ta, S, dtype=torch.long, device=device), unres=torch.ones(Ta, dtype=torch.bool, device=device), alpha=al, nbrs=torch.zeros(S, 1, dtype=torch.long, device=device), **kw)
    tt = time.time() - t0; best = int(torch.argmin(v)); yh = predict(m, sp["Xte"]).cpu().numpy()
    sens = [dict(alpha=a_, val_rmse=float(np.sqrt(float(v[i])) * sp["sd"]), test_rmse=score(yh[i], sp)[0]["rmse"]) for i, a_ in enumerate(al)]
    finish("Mixup", m, v, tt, st, t_idx=best, extra=dict(selected=al[best], sensitivity=sens))
    # neighbours (shared by Global kNN and RegMix)
    t0 = time.time(); opts = sorted({min(o, S - 1) for o in CFG["knn_options"]}); nbrs = knn_table(Xtr, max(opts)); t_nb = time.time() - t0
    # Global kNN: one k for every example (k from the same option set, k >= 1)
    ko = [o for o in opts if o >= 1]; Tk = len(ko); t0 = time.time()
    m, v, st = train_batched(Xtr, ytr, Xva, yva, T=Tk, ks=torch.tensor(ko, device=device)[:, None].expand(Tk, S).contiguous(), alpha=CFG["regmix_alpha"], nbrs=nbrs, **kw)
    tt = time.time() - t0 + t_nb; best = int(torch.argmin(v)); yh = predict(m, sp["Xte"]).cpu().numpy()
    sens = [dict(k=k_, val_rmse=float(np.sqrt(float(v[i])) * sp["sd"]), test_rmse=score(yh[i], sp)[0]["rmse"]) for i, k_ in enumerate(ko)]
    finish("GlobalkNN", m, v, tt, st, t_idx=best, extra=dict(selected=ko[best], sensitivity=sens))
    # RegMix
    if protocol == "chrono" and S > CFG["chrono_regmix_max_n"]: print(f"  RegMix skipped for n={S} (> chrono_regmix_max_n)")
    else:
        t0 = time.time(); ks_final, hist = regmix_search(Xtr, ytr, Xva, yva, nbrs, opts, seed, lr, CFG["regmix_iters"], CFG["search_steps"]); t_search = time.time() - t0 + t_nb
        t0 = time.time(); m, v, st = train_batched(Xtr, ytr, Xva, yva, T=1, ks=ks_final[None], alpha=CFG["regmix_alpha"], nbrs=nbrs, **kw); t_fin = time.time() - t0
        counts = {str(o): float((ks_final == o).float().mean()) for o in opts}
        finish("RegMix", m, v, t_search + t_fin, st, extra=dict(policy_hist=counts, search_hist=hist[-3:]), search_time=t_search)
    return rows

done = set()
if os.path.exists(RESULTS):
    for line in open(RESULTS): r = json.loads(line); done.add((r["protocol"], r["frac"], r["seed"]))
t_all = time.time()
for protocol in PROTOCOLS:
    lr = select_lr(protocol)
    for frac in FRACS:
        for seed in SEEDS:
            if (protocol, frac, seed) in done: continue
            t0 = time.time(); rows = run_one(protocol, frac, seed, lr)
            with open(RESULTS, "a") as fh:
                for r in rows: fh.write(json.dumps(r) + "\n")
            print(f"[{protocol} frac={frac} seed={seed}] " + " | ".join(f"{r['method']} RMSE={r['rmse']:.2f}" for r in rows) + f"  ({time.time() - t0:.0f}s)")
print("total wall-clock this session: %.1f min" % ((time.time() - t_all) / 60))

In [ ]:
# 9. Tables: main comparison (Table 3), size sweep (Table 4), sensitivity, and statistics
R = pd.DataFrame([json.loads(l) for l in open(RESULTS)]); R["total_time_s"] = R["time_s"]
METHODS = ["Vanilla", "Mixup", "GlobalkNN", "RegMix"]; ms = lambda s: f"{s.mean():.2f} ± {s.std(ddof=1):.2f}" if len(s) > 1 else f"{s.mean():.2f}"
for pr in PROTOCOLS:
    d = R[(R.protocol == pr) & (R.frac == 1.0)]
    t3 = pd.DataFrame({m: dict(RMSE=ms(g.rmse), MAE=ms(g.mae), R2=f"{g.r2.mean():.3f} ± {g.r2.std(ddof=1):.3f}", **{"Training time (s)": ms(g.total_time_s)}, Runs=len(g)) for m, g in d.groupby("method")}).T.reindex([m for m in METHODS if m in d.method.unique()])
    print(f"\nTable 3 [{pr}] n_train={int(d.n_train.iloc[0])}; RMSE/MAE in bikes (test set); mean ± sd over seeds"); display(t3); t3.to_csv(os.path.join(OUT, "tables", f"table3_main_{pr}.csv"))
    g4 = R[R.protocol == pr].groupby(["frac", "n_train", "method"]).rmse.agg(["mean", "std"]); g4["s"] = g4.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}" if r["std"] == r["std"] else f"{r['mean']:.2f}", axis=1)
    t4 = g4["s"].unstack("method").reindex(columns=[m for m in METHODS if m in R.method.unique()])
    print(f"Table 4 [{pr}] test RMSE by training-set size"); display(t4); t4.to_csv(os.path.join(OUT, "tables", f"table4_size_{pr}.csv"))

def holm(p):
    p = np.asarray(p); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rank, i in enumerate(o): run = max(run, (len(p) - rank) * p[i]); adj[i] = min(1.0, run)
    return adj
COMPS = [("Vanilla", "Mixup"), ("Vanilla", "GlobalkNN"), ("Vanilla", "RegMix"), ("Mixup", "RegMix"), ("GlobalkNN", "RegMix")]
rng = np.random.RandomState(0); srows = []
for pr in PROTOCOLS:
    for fr in FRACS:
        piv = R[(R.protocol == pr) & (R.frac == fr)].pivot(index="seed", columns="method", values="rmse")
        for a, b in COMPS:
            if a not in piv or b not in piv: continue
            x = piv[[a, b]].dropna(); n = len(x)
            if n < 3: continue
            diff = x[a] - x[b]            # > 0  => second method has LOWER RMSE (better)
            bs = [diff.values[rng.randint(0, n, n)].mean() for _ in range(5000)]
            try: pw = stats.wilcoxon(diff).pvalue
            except ValueError: pw = np.nan
            srows.append(dict(protocol=pr, frac=fr, n_pairs=n, A=a, B=b, mean_rmse_gain_of_B=diff.mean(), ci95_low=np.percentile(bs, 2.5), ci95_high=np.percentile(bs, 97.5),
                              cohen_dz=diff.mean() / (diff.std(ddof=1) + 1e-12), p_ttest=stats.ttest_rel(x[a], x[b]).pvalue, p_wilcoxon=pw))
S_ = pd.DataFrame(srows)
if len(S_):
    S_["p_ttest_holm"] = holm(S_.p_ttest.values); S_.to_csv(os.path.join(OUT, "tables", "paired_stats.csv"), index=False)
    print("\nPaired comparisons (positive gain = B has lower RMSE than A). With <6 pairs a two-sided Wilcoxon test cannot reach p<0.05; treat p-values as descriptive."); display(S_.round(4))

sens_rows = []
for _, r in R[(R.frac == 1.0)].iterrows():
    if isinstance(r.get("sensitivity"), list):
        for s in r["sensitivity"]: sens_rows.append(dict(protocol=r.protocol, method=r.method, param=s.get("alpha", s.get("k")), seed=r.seed, test_rmse=s["test_rmse"], val_rmse=s["val_rmse"]))
if sens_rows:
    SN = pd.DataFrame(sens_rows).groupby(["protocol", "method", "param"])[["val_rmse", "test_rmse"]].mean().round(2)
    print("\nHyperparameter sensitivity (mean over seeds, full training set): Mixup alpha, Global-kNN k"); display(SN); SN.to_csv(os.path.join(OUT, "tables", "sensitivity.csv"))

In [ ]:
# 10. Figures 4-9
PAL = {"Vanilla": "#8172B2", "Mixup": "#CCB974", "GlobalkNN": "#64B5CD", "RegMix": "#C44E52"}
for pr in PROTOCOLS:   # Fig 4 & 5: actual vs predicted, residuals (seed 0, full training set)
    fs = {m: os.path.join(OUT, "preds", f"{pr}_{m}.npy") for m in METHODS if os.path.exists(os.path.join(OUT, "preds", f"{pr}_{m}.npy"))}
    if not fs: continue
    y = get_split(pr, 1.0, SEEDS[0])["yte_raw"]; fig, ax = plt.subplots(1, len(fs), figsize=(4 * len(fs), 3.8), squeeze=False)
    for a, (m, f) in zip(ax[0], fs.items()):
        yh = np.load(f); a.scatter(y, yh, s=4, alpha=.25, color=PAL[m]); lim = [0, max(y.max(), yh.max())]; a.plot(lim, lim, "k--", lw=1); a.set(xlabel="Actual rentals [bikes/h]", ylabel="Predicted rentals", title=m)
    plt.suptitle(f"Actual vs. predicted demand ({pr} protocol, seed {SEEDS[0]}, test set)", y=1.03); plt.tight_layout(); savefig(f"fig4_actual_vs_predicted_{pr}")
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    for m, f in fs.items():
        res = y - np.load(f); sns.kdeplot(res, ax=ax[0], label=m, color=PAL[m]); ax[1].scatter(np.load(f), res, s=3, alpha=.15, color=PAL[m], label=m)
    ax[0].set(xlabel="Residual (actual - predicted) [bikes/h]", ylabel="Density", title="(a) Residual distribution"); ax[0].legend()
    ax[1].axhline(0, color="k", lw=1); ax[1].set(xlabel="Predicted rentals", ylabel="Residual", title="(b) Residuals vs. prediction"); plt.tight_layout(); savefig(f"fig5_residuals_{pr}")

fig, ax = plt.subplots(1, len(PROTOCOLS), figsize=(5.5 * len(PROTOCOLS), 3.8), squeeze=False)   # Fig 6
for a, pr in zip(ax[0], PROTOCOLS):
    d = R[(R.protocol == pr) & (R.frac == 1.0)]; g = d.groupby("method").rmse.agg(["mean", "std"]).reindex([m for m in METHODS if m in d.method.unique()])
    a.bar(g.index, g["mean"], yerr=g["std"], color=[PAL[m] for m in g.index], capsize=4); a.set(ylabel="Test RMSE [bikes/h]", title=f"{pr} protocol (n_train={int(d.n_train.iloc[0])})")
plt.tight_layout(); savefig("fig6_rmse_comparison")

fig, ax = plt.subplots(1, len(PROTOCOLS), figsize=(5.8 * len(PROTOCOLS), 4), squeeze=False)    # Fig 7
for a, pr in zip(ax[0], PROTOCOLS):
    for m in METHODS:
        g = R[(R.protocol == pr) & (R.method == m)].groupby("n_train").rmse.agg(["mean", "std"]).dropna(subset=["mean"])
        if len(g): a.errorbar(g.index, g["mean"], yerr=g["std"], marker="o", capsize=3, label=m, color=PAL[m])
    a.set(xscale="log", xlabel="Training-set size (examples, log scale)", ylabel="Test RMSE [bikes/h]", title=f"{pr} protocol"); a.legend()
plt.tight_layout(); savefig("fig7_performance_vs_size")

fig, ax = plt.subplots(1, len(PROTOCOLS), figsize=(5.5 * len(PROTOCOLS), 3.8), squeeze=False)  # Fig 8
for a, pr in zip(ax[0], PROTOCOLS):
    d = R[(R.protocol == pr) & (R.frac == 1.0)]; sns.boxplot(data=d, x="method", y="rmse", order=[m for m in METHODS if m in d.method.unique()], palette=PAL, ax=a, showfliers=False)
    sns.stripplot(data=d, x="method", y="rmse", order=[m for m in METHODS if m in d.method.unique()], color="k", size=3, ax=a); a.set(xlabel="", ylabel="Test RMSE [bikes/h]", title=f"{pr}: variability across seeds")
plt.tight_layout(); savefig("fig8_seed_variability")

ph = R[(R.method == "RegMix") & (R.frac == 1.0) & R.policy_hist.notna()] if "policy_hist" in R else R.iloc[0:0]   # Fig 9a: learned kNN option frequencies (cf. paper Fig. 6)
if len(ph):
    fig, ax = plt.subplots(1, len(ph.protocol.unique()), figsize=(5 * len(ph.protocol.unique()), 3.4), squeeze=False)
    for a, (pr, g) in zip(ax[0], ph.groupby("protocol")):
        H = pd.DataFrame(list(g.policy_hist)).mean(); a.bar(H.index.astype(str), 100 * H.values, color=PAL["RegMix"]); a.set(xlabel="kNN option (k; 0 = no mixing)", ylabel="Share of examples [%]", title=f"{pr}: learned policy (mean over seeds)")
    plt.tight_layout(); savefig("fig9a_regmix_policy_histogram")

# Fig 9b: label consistency vs. mixing distance (idea of paper Sec. 4.3): how far does the interpolated label deviate from a Vanilla model's prediction at the mixed input?
sp = get_split("paper", 1.0, SEEDS[0]); lr0 = select_lr("paper")
m0, _, _ = train_batched(sp["Xtr"], sp["ytr"], sp["Xva"], sp["yva"], T=1, seed=SEEDS[0], lr=lr0, max_steps=CFG["max_steps"], eval_every=CFG["eval_every"], patience=CFG["patience"])
g = torch.Generator(device=device).manual_seed(0); S0 = sp["n"]; i = torch.randint(0, S0, (20000,), device=device, generator=g); j = torch.randint(0, S0, (20000,), device=device, generator=g)
dist = (sp["Xtr"][i] - sp["Xtr"][j]).norm(dim=1); xm = 0.5 * sp["Xtr"][i] + 0.5 * sp["Xtr"][j]; ym = 0.5 * sp["ytr"][i] + 0.5 * sp["ytr"][j]
err = ((predict(m0, xm)[0] - ym).abs() * sp["sd"]).cpu().numpy(); dd = pd.DataFrame(dict(dist=dist.cpu().numpy(), err=err)); dd["bin"] = pd.qcut(dd.dist, 10)
b = dd.groupby("bin").agg(dist=("dist", "mean"), err=("err", "mean")); plt.figure(figsize=(5.5, 3.8)); plt.plot(b.dist, b.err, "o-", color=PAL["Mixup"])
plt.xlabel("Euclidean distance between mixed pair (standardised features)"); plt.ylabel("Mean |interpolated label - Vanilla prediction| [bikes/h]"); plt.title("Label consistency vs. mixing distance (λ=0.5)"); savefig("fig9b_label_consistency_vs_distance")
print("Caution: this diagnostic uses an imperfect Vanilla model as a reference, as in the paper's Sec. 4.3; read the trend, not the absolute values.")

In [ ]:
# 11. Save configuration, environment and a zip of all outputs
env = dict(python=platform.python_version(), torch=torch.__version__, numpy=np.__version__, pandas=pd.__version__, device=str(device),
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, data_sha256=sha, seeds=SEEDS)
json.dump(dict(config=CFG, environment=env), open(os.path.join(OUT, "config.json"), "w"), indent=2, default=str)
table2 = pd.DataFrame({"Setting": list(CFG.keys()) + list(env.keys()), "Value": [str(v) for v in CFG.values()] + [str(v) for v in env.values()]})
table2.to_csv(os.path.join(OUT, "tables", "table2_configuration.csv"), index=False); display(table2)
import shutil; zip_path = shutil.make_archive(os.path.join(os.path.dirname(OUT.rstrip("/")), "bike_regmix_outputs"), "zip", OUT); print("zip:", zip_path)
if IN_COLAB:
    from google.colab import files; files.download(zip_path)
print("Done. Paste the Table 3 / Table 4 / paired-stats outputs back into the chat to write Results, Discussion and Abstract.")